In [2]:
!pip install torch numpy scipy scikit-learn -q

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import random

print("✅ Packages installed and imported")

✅ Packages installed and imported


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/reddit"

feats = np.load(f"{DATA_PATH}/reddit-feats.npy")
with open(f"{DATA_PATH}/reddit-G.json") as f:
    G_data = json.load(f)
with open(f"{DATA_PATH}/reddit-id_map.json") as f:
    id_map = json.load(f)
with open(f"{DATA_PATH}/reddit-class_map.json") as f:
    class_map = json.load(f)

print(f"✅ Reddit data loaded successfully!")
print(f"   Nodes: {feats.shape[0]:,}")
print(f"   Feature dim: {feats.shape[1]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Reddit data loaded successfully!
   Nodes: 232,965
   Feature dim: 602


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device} (GPU enabled!)")

✅ Using device: cpu (GPU enabled!)


In [5]:
class GraphSAGE_Pool(nn.Module):
    def __init__(self, node_in_dim, neighbor_in_dim, hidden_dim):
        super().__init__()
        self.mlp = nn.Linear(neighbor_in_dim, hidden_dim)
        self.linear = nn.Linear(node_in_dim + hidden_dim, hidden_dim)

    def forward(self, x, neigh):
        neigh = F.relu(self.mlp(neigh))
        h_neigh, _ = torch.max(neigh, dim=1)          # max-pool
        h = torch.cat([x, h_neigh], dim=1)
        return F.relu(self.linear(h))

In [6]:
class GraphSAGE_LSTM(nn.Module):
    def __init__(self, node_in_dim, neighbor_in_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(neighbor_in_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(node_in_dim + hidden_dim, hidden_dim)

    def forward(self, x, neigh):
        # Random permutation (as in original GraphSAGE paper)
        perm = torch.randperm(neigh.size(1), device=neigh.device)
        neigh = neigh[:, perm, :]

        _, (h_n, _) = self.lstm(neigh)
        h_neigh = h_n.squeeze(0)
        h = torch.cat([x, h_neigh], dim=1)
        return F.relu(self.linear(h))

In [12]:
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes, agg="pool", dropout=0.5):
        super().__init__()
        self.dropout = dropout

        if agg == "lstm":
            self.layer1 = GraphSAGE_LSTM(in_dim, in_dim, hidden_dim)
            self.layer2 = GraphSAGE_LSTM(hidden_dim, in_dim, hidden_dim)
        else:  # pool
            self.layer1 = GraphSAGE_Pool(in_dim, in_dim, hidden_dim)
            self.layer2 = GraphSAGE_Pool(hidden_dim, in_dim, hidden_dim)

        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.dropout_layer = nn.Dropout(p=dropout)

    def forward(self, x, neigh):
        h1 = self.layer1(x, neigh)
        h1 = self.dropout_layer(h1)
        h2 = self.layer2(h1, neigh)
        h2 = self.dropout_layer(h2)
        return h2

    def predict(self, x, neigh):
        h = self.forward(x, neigh)
        return self.classifier(h)

In [22]:
# ========================== HARDER DUMMY DATA (more realistic) ==========================
N = 2000                    # increased size
F_dim = 128
num_classes = 10            # more classes
num_neighbors = 15

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Overlapping clusters with high noise
centers = torch.randn(num_classes, F_dim, device=device) * 1.5

x = torch.zeros(N, F_dim, device=device)
labels = torch.zeros(N, dtype=torch.long, device=device)

for i in range(N):
    cls = i % num_classes
    labels[i] = cls
    # Much higher noise + slight overlap between classes
    x[i] = centers[cls] + torch.randn(F_dim, device=device) * 2.2

# Weak homophily (only 30% same-class neighbors) — realistic for social networks
neighbors = torch.zeros(N, num_neighbors, F_dim, device=device)
for i in range(N):
    cls = labels[i].item()
    same_class = torch.where(labels == cls)[0]
    other_class = torch.where(labels != cls)[0]

    num_same = int(num_neighbors * 0.30)          # only 30% same class
    same_idx = same_class[torch.randint(0, len(same_class), (num_same,))]
    other_idx = other_class[torch.randint(0, len(other_class), (num_neighbors - num_same,))]

    neigh_idx = torch.cat([same_idx, other_idx])
    neighbors[i] = x[neigh_idx]

print("✅ Harder dummy data created (weak homophily + high noise)")
print("   → Same-class neighbors = 30% only")
print("   → Expected realistic scores: Supervised ~0.75-0.85 | Unsupervised ~0.65-0.78")

✅ Harder dummy data created (weak homophily + high noise)
   → Same-class neighbors = 30% only
   → Expected realistic scores: Supervised ~0.75-0.85 | Unsupervised ~0.65-0.78


In [23]:
# Train/test split
idx = np.arange(N)
train_idx, test_idx = train_test_split(idx, test_size=0.3, random_state=42)
train_idx = torch.LongTensor(train_idx).to(device)
test_idx = torch.LongTensor(test_idx).to(device)

def train_supervised(model, x, neighbors, labels, epochs=20, lr=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        out = model.predict(x, neighbors)

        loss = F.cross_entropy(out[train_idx], labels[train_idx])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Evaluation
        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_f1 = f1_score(labels[train_idx].cpu().numpy(),
                                pred[train_idx].cpu().numpy(), average='micro')
            test_f1 = f1_score(labels[test_idx].cpu().numpy(),
                               pred[test_idx].cpu().numpy(), average='micro')

        if epoch % 5 == 0 or epoch == epochs-1:
            print(f"[SUP {agg.upper()}] Epoch {epoch:2d} | Loss: {loss.item():.4f} | "
                  f"Train F1: {train_f1:.4f} | Test F1: {test_f1:.4f}")

    return test_f1

In [24]:
def unsup_loss(z_u, z_v, z_neg, temperature=0.2):
    # CRITICAL FIX: L2 normalize embeddings (prevents explosion)
    z_u = F.normalize(z_u, dim=1)
    z_v = F.normalize(z_v, dim=1)
    z_neg = F.normalize(z_neg, dim=1)

    # Scaled similarities
    pos_sim = torch.sum(z_u * z_v, dim=1) / temperature
    neg_sim = torch.sum(z_u * z_neg, dim=1) / temperature

    # Numerically stable loss (equivalent to original but safe)
    loss = -torch.log(torch.sigmoid(pos_sim)) - torch.log(torch.sigmoid(-neg_sim))
    return loss.mean()


def train_unsupervised(model, x, neighbors, labels, epochs=30, lr=0.0005):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    N = x.size(0)

    for epoch in range(epochs):
        model.train()
        z = model(x, neighbors)                     # [N, hidden_dim]

        # Graph-aware sampling (positive = real neighbor, negative = random)
        neigh_idx = torch.randint(0, neighbors.size(1), (N,), device=device)
        pos_idx = torch.arange(N, device=device)
        neg_idx = torch.randint(0, N, (N,), device=device)

        z_u = z[pos_idx]
        z_v = z[neigh_idx]      # actual neighbor (70% same class)
        z_n = z[neg_idx]        # random negative

        loss = unsup_loss(z_u, z_v, z_n)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 5 == 0 or epoch == epochs-1:
            print(f"[UNSUP {agg.upper()}] Epoch {epoch:2d} | Loss: {loss.item():.4f}")

    # Final downstream evaluation
    model.eval()
    with torch.no_grad():
        z = model(x, neighbors).cpu().numpy()
    y = labels.cpu().numpy()

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(z, y)
    pred = clf.predict(z)
    f1 = f1_score(y, pred, average='micro')
    print(f"→ Unsupervised downstream F1: {f1:.4f}\n")
    return f1

In [25]:
print("="*70)
print("🚀 TRAINING GraphSAGE POOL")
agg = "pool"
model_pool = GraphSAGE(F_dim, 128, num_classes, agg=agg, dropout=0.5).to(device)
test_f1_pool = train_supervised(model_pool, x, neighbors, labels, epochs=40, lr=0.0005)
unsup_f1_pool = train_unsupervised(model_pool, x, neighbors, labels)

print("="*70)
print("🚀 TRAINING GraphSAGE LSTM")
agg = "lstm"
model_lstm = GraphSAGE(F_dim, 128, num_classes, agg=agg, dropout=0.5).to(device)
test_f1_lstm = train_supervised(model_lstm, x, neighbors, labels, epochs=40, lr=0.0005)
unsup_f1_lstm = train_unsupervised(model_lstm, x, neighbors, labels)

print("="*70)
print("🎉 FINAL SUMMARY (After Graph-Aware Unsupervised)")
print(f"POOL  → Supervised Test F1: {test_f1_pool:.4f} | Unsup F1: {unsup_f1_pool:.4f}")
print(f"LSTM  → Supervised Test F1: {test_f1_lstm:.4f} | Unsup F1: {unsup_f1_lstm:.4f}")

🚀 TRAINING GraphSAGE POOL
[SUP POOL] Epoch  0 | Loss: 2.4193 | Train F1: 0.1171 | Test F1: 0.1083
[SUP POOL] Epoch  5 | Loss: 2.1324 | Train F1: 0.2357 | Test F1: 0.2567
[SUP POOL] Epoch 10 | Loss: 1.8847 | Train F1: 0.4036 | Test F1: 0.3983
[SUP POOL] Epoch 15 | Loss: 1.5728 | Train F1: 0.5421 | Test F1: 0.4800
[SUP POOL] Epoch 20 | Loss: 1.2433 | Train F1: 0.6671 | Test F1: 0.6100
[SUP POOL] Epoch 25 | Loss: 0.9308 | Train F1: 0.7629 | Test F1: 0.7033
[SUP POOL] Epoch 30 | Loss: 0.6479 | Train F1: 0.8486 | Test F1: 0.8133
[SUP POOL] Epoch 35 | Loss: 0.4505 | Train F1: 0.9007 | Test F1: 0.8917
[SUP POOL] Epoch 39 | Loss: 0.3471 | Train F1: 0.9300 | Test F1: 0.9233
[UNSUP POOL] Epoch  0 | Loss: 1.9090
[UNSUP POOL] Epoch  5 | Loss: 1.5840
[UNSUP POOL] Epoch 10 | Loss: 1.4869
[UNSUP POOL] Epoch 15 | Loss: 1.4724
[UNSUP POOL] Epoch 20 | Loss: 1.4377
[UNSUP POOL] Epoch 25 | Loss: 1.4400
[UNSUP POOL] Epoch 29 | Loss: 1.4345
→ Unsupervised downstream F1: 0.8120

🚀 TRAINING GraphSAGE LSTM
[SU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
